In [34]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.metrics import roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay

import optuna
from optuna.samplers import TPESampler

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


In [35]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name="Model", normalize_cm='true'):
    """
    Plots ROC curves for train and test sets, and a confusion matrix for the test set.
    
    Parameters
    ----------
    model : fitted estimator
        Any sklearn-compatible classifier with predict() and predict_proba() or decision_function().
    X_train, y_train, X_test, y_test : arrays or DataFrames
        Training and testing datasets.
    model_name : str
        Name of the model (for plot titles).
    normalize_cm : {'true', 'pred', 'all', None}
        Normalization mode for confusion matrix.
    """
    # --- Get scores for train/test ---
    if hasattr(model, "predict_proba"):
        y_train_score = model.predict_proba(X_train)[:, 1]
        y_test_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_train_score = model.decision_function(X_train)
        y_test_score = model.decision_function(X_test)
    else:
        raise ValueError(f"{model_name} does not support probability or decision_function output.")

    # --- Compute AUCs ---
    auc_train = roc_auc_score(y_train, y_train_score)
    auc_test = roc_auc_score(y_test, y_test_score)

    print(f"\n=== {model_name} ===")
    print(f"Train ROC AUC: {auc_train:.4f}")
    print(f"Test ROC AUC:  {auc_test:.4f}")

    # --- Plot ROC curves for Train vs Test ---
    fig, ax = plt.subplots(figsize=(6, 6))
    RocCurveDisplay.from_predictions(y_train, y_train_score, name="Train", ax=ax)
    RocCurveDisplay.from_predictions(y_test, y_test_score, name="Test", ax=ax)
    plt.title(f"ROC Curves — {model_name}\nTrain AUC={auc_train:.3f} | Test AUC={auc_test:.3f}")
    plt.plot([0, 1], [0, 1], "k--", label="Chance")
    plt.legend()
    plt.show()

    # --- Confusion Matrix for test set only ---
    y_pred_test = model.predict(X_test)
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, normalize=normalize_cm)
    plt.title(f"Confusion Matrix — {model_name} (normalize={normalize_cm})")
    plt.show()

In [36]:
df = pd.read_csv('../data/risk-model-dataset.csv')
df.head()

,Applicant_ID,Annual_Income,Applicant_Age,Work_Experience,Marital_Status,House_Ownership,Vehicle_Ownership(car),Occupation,Residence_City,Residence_State,Years_in_Current_Employment,Years_in_Current_Residence,Loan_Default_Risk
0,75722,9657655,76,0,single,rented,no,Psychologist,Jalandhar,Punjab,0,12,0
1,80185,9259353,37,18,single,rented,no,Petroleum_Engineer,Bally,West_Bengal,12,11,0
2,19865,1509721,66,8,single,rented,no,Drafter,Indore,Madhya_Pradesh,4,12,0
3,76700,5867312,43,1,single,owned,no,Chartered_Accountant,Kurnool[18],Andhra_Pradesh,1,13,1
4,92992,7223191,44,9,single,rented,no,Air_traffic_controller,Asansol,West_Bengal,9,13,0


In [37]:
rem_cols = ['Applicant_ID']
df = df.drop(columns=rem_cols)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 12 columns):
 #   Column                       Non-Null Count   Dtype 
---  ------                       --------------   ----- 
 0   Annual_Income                100000 non-null  int64 
 1   Applicant_Age                100000 non-null  int64 
 2   Work_Experience              100000 non-null  int64 
 3   Marital_Status               100000 non-null  object
 4   House_Ownership              100000 non-null  object
 5   Vehicle_Ownership(car)       100000 non-null  object
 6   Occupation                   100000 non-null  object
 7   Residence_City               100000 non-null  object
 8   Residence_State              100000 non-null  object
 9   Years_in_Current_Employment  100000 non-null  int64 
 10  Years_in_Current_Residence   100000 non-null  int64 
 11  Loan_Default_Risk            100000 non-null  int64 
dtypes: int64(6), object(6)
memory usage: 9.2+ MB


In [38]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols.remove('Loan_Default_Risk')

In [50]:
X = df.drop('Loan_Default_Risk', axis=1)
y = df['Loan_Default_Risk']

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat = ohe.fit_transform(X[cat_cols])

X_cat_df = pd.DataFrame(
    X_cat,
    columns=ohe.get_feature_names_out(cat_cols),
    index=X.index
)


X = pd.concat([X[num_cols], X_cat_df], axis=1)
X.columns = X.columns.str.replace(r"[\[\]]", "", regex=True)
X.head()

,Annual_Income,Applicant_Age,Work_Experience,Years_in_Current_Employment,Years_in_Current_Residence,Marital_Status_married,Marital_Status_single,House_Ownership_norent_noown,House_Ownership_owned,House_Ownership_rented,...,Residence_State_Punjab,Residence_State_Rajasthan,Residence_State_Sikkim,Residence_State_Tamil_Nadu,Residence_State_Telangana,Residence_State_Tripura,Residence_State_Uttar_Pradesh,Residence_State_Uttar_Pradesh5,Residence_State_Uttarakhand,Residence_State_West_Bengal
0,9657655,76,0,0,12,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,9259353,37,18,12,11,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,1509721,66,8,4,12,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,5867312,43,1,1,13,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,7223191,44,9,9,13,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [51]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify = y
)

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])


In [41]:
CV = KFold(n_splits=5, shuffle=True, random_state=42)
N_TRIALS = 20
SAMPLER = TPESampler(seed=42)

In [42]:
# --- Logistic Regression: tune with Bayesian optimization (TPE), then fit best on X_train ---
def objective_lr(trial):
    penalty = trial.suggest_categorical("penalty", ["l2", "l1"])
    clf = LogisticRegression(
        C=trial.suggest_float("C", 1e-4, 1e2, log=True),
        penalty=penalty,
        solver="saga",          # supports l1/l2 and sparse data
        class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
        max_iter=5000,
        n_jobs=-1,
        random_state=42,
    )
    scores = cross_val_score(clf, X_train, y_train, cv=CV, scoring="roc_auc", n_jobs=-1)
    return float(np.mean(scores))

study_lr = optuna.create_study(direction="maximize", sampler=SAMPLER)
study_lr.optimize(objective_lr, n_trials=N_TRIALS, show_progress_bar=False)

best_lr = LogisticRegression(
    **{k: v for k, v in study_lr.best_params.items()},
    solver="saga", max_iter=5000, n_jobs=-1, random_state=42
)
best_lr.fit(X_train, y_train)
print("LogReg best AUC:", study_lr.best_value)
print("LogReg best params:", study_lr.best_params)

[I 2025-11-12 01:26:08,416] A new study created in memory with name: no-name-a416687c-d410-4e90-83ca-16739b5a0c16
[I 2025-11-12 01:26:33,851] Trial 0 finished with value: 0.6712372453940407 and parameters: {'penalty': 'l1', 'C': 2.4658329458549115, 'class_weight': None}. Best is trial 0 with value: 0.6712372453940407.
[I 2025-11-12 01:26:47,213] Trial 1 finished with value: 0.6706763378958157 and parameters: {'penalty': 'l2', 'C': 15.741890047456641, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.6712372453940407.
[I 2025-11-12 01:28:19,760] Trial 2 finished with value: 0.6711652765375451 and parameters: {'penalty': 'l1', 'C': 9.877700294007917, 'class_weight': None}. Best is trial 0 with value: 0.6712372453940407.
[I 2025-11-12 01:28:23,140] Trial 3 finished with value: 0.6609121117225938 and parameters: {'penalty': 'l1', 'C': 0.14077923139972404, 'class_weight': None}. Best is trial 0 with value: 0.6712372453940407.
[I 2025-11-12 01:28:26,614] Trial 4 finished with value:

LogReg best AUC: 0.6712738962940763
LogReg best params: {'penalty': 'l1', 'C': 1.0524814224284538, 'class_weight': None}


In [43]:
# --- Random Forest: tune with Bayesian optimization (TPE), then fit best on X_train ---
def objective_rf(trial):
    clf = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 600),
        max_depth=trial.suggest_int("max_depth", 3, 40),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        bootstrap=trial.suggest_categorical("bootstrap", [True, False]),
        class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
        n_jobs=-1,
        random_state=42,
    )
    scores = cross_val_score(clf, X_train, y_train, cv=CV, scoring="roc_auc", n_jobs=-1)
    return float(np.mean(scores))

study_rf = optuna.create_study(direction="maximize", sampler=SAMPLER)
study_rf.optimize(objective_rf, n_trials=N_TRIALS, show_progress_bar=False)

best_rf = RandomForestClassifier(random_state=42, n_jobs=-1, **study_rf.best_params)
best_rf.fit(X_train, y_train)
print("RF best AUC:", study_rf.best_value)
print("RF best params:", study_rf.best_params)

[I 2025-11-12 01:35:18,703] A new study created in memory with name: no-name-ecbd7f50-37e0-4e1b-ae3d-98cd89d4de1d
[I 2025-11-12 01:35:25,858] Trial 0 finished with value: 0.8424723345222361 and parameters: {'n_estimators': 585, 'max_depth': 32, 'min_samples_split': 19, 'min_samples_leaf': 9, 'max_features': 'log2', 'bootstrap': True, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8424723345222361.
[I 2025-11-12 01:38:12,268] Trial 1 finished with value: 0.7930980488312812 and parameters: {'n_estimators': 235, 'max_depth': 34, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_features': None, 'bootstrap': False, 'class_weight': None}. Best is trial 0 with value: 0.8424723345222361.
[I 2025-11-12 01:38:16,910] Trial 2 finished with value: 0.9497766874349045 and parameters: {'n_estimators': 102, 'max_depth': 33, 'min_samples_split': 15, 'min_samples_leaf': 8, 'max_features': 'sqrt', 'bootstrap': False, 'class_weight': None}. Best is trial 2 with value: 0.9497766874349045.
[I

RF best AUC: 0.963686038585059
RF best params: {'n_estimators': 347, 'max_depth': 40, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': False, 'class_weight': None}


In [54]:
# --- XGBoost: tune with Bayesian optimization (TPE), then fit best on X_train ---
def objective_xgb(trial):
    clf = XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 800),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        min_child_weight=trial.suggest_float("min_child_weight", 0.1, 10.0, log=True),
        gamma=trial.suggest_float("gamma", 0.0, 5.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        tree_method=trial.suggest_categorical("tree_method", ["hist", "approx", "exact"]),
        objective="binary:logistic",
        eval_metric="auc",
        n_jobs=-1,
        random_state=42,
    )
    scores = cross_val_score(clf, X_train, y_train, cv=CV, scoring="roc_auc", n_jobs=-1)
    return float(np.mean(scores))

study_xgb = optuna.create_study(direction="maximize", sampler=SAMPLER)
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=False)

best_xgb = XGBClassifier(
    n_jobs=-1, random_state=42, objective="binary:logistic", eval_metric="auc",
    **study_xgb.best_params
)
best_xgb.fit(X_train, y_train)
print("XGB best AUC:", study_xgb.best_value)
print("XGB best params:", study_xgb.best_params)

[I 2025-11-12 01:53:41,605] A new study created in memory with name: no-name-c8c7efca-e2be-4c08-978d-6b8f444edefc
[W 2025-11-12 01:55:34,760] Trial 0 failed with parameters: {'n_estimators': 543, 'max_depth': 6, 'learning_rate': 0.007328826823009889, 'subsample': 0.8629778394351197, 'colsample_bytree': 0.9485551299762885, 'min_child_weight': 5.945287330479987, 'gamma': 3.899377729288119, 'reg_alpha': 0.0060018455525164905, 'reg_lambda': 5.718204526246171e-08, 'tree_method': 'approx'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/opt/anaconda3/envs/capstone_env/lib/python3.12/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/s2/hqtffcyx5bq11_ppcq5qvcgc0000gn/T/ipykernel_14806/3713091380.py", line 19, in objective_xgb
    scores = cross_val_score(clf, X_train, y_train, cv=CV, scoring="roc_auc", n_jobs=-1)
             ^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

In [ ]:
import joblib

# Example: after you’ve trained your best model
joblib.dump(best_xgb, "../models/best_xgb_model.joblib")
joblib.dump(best_rf, "../models/best_random_forest.joblib")
joblib.dump(best_lr, "../models/best_logreg_model.joblib")

In [55]:
evaluate_model(best_lr, X_train, y_train, X_test, y_test, model_name="Logistic Regression")

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- Residence_City_Anantapuram24
- Residence_City_Aurangabad39
- Residence_City_Bettiah33
- Residence_City_Buxar37
- Residence_City_Chittoor28
- ...
Feature names seen at fit time, yet now missing:
- Residence_City_Anantapuram[24]
- Residence_City_Aurangabad[39]
- Residence_City_Bettiah[33]
- Residence_City_Buxar[37]
- Residence_City_Chittoor[28]
- ...


In [56]:
evaluate_model(best_rf, X_train, y_train, X_test, y_test, model_name="Random Forest")

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- Residence_City_Anantapuram24
- Residence_City_Aurangabad39
- Residence_City_Bettiah33
- Residence_City_Buxar37
- Residence_City_Chittoor28
- ...
Feature names seen at fit time, yet now missing:
- Residence_City_Anantapuram[24]
- Residence_City_Aurangabad[39]
- Residence_City_Bettiah[33]
- Residence_City_Buxar[37]
- Residence_City_Chittoor[28]
- ...


In [ ]:
evaluate_model(best_xgb, X_train, y_train, X_test, y_test, model_name="XGBoost")